In [97]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np 
import pandas as pd 
import kagglehub
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.metrics import brier_score_loss

import kagglehub

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os








DATA_PATH = '/kaggle/input/competitions/march-machine-learning-mania-2026/' #just to make it look cleaner

m_teams = pd.read_csv(DATA_PATH + 'MTeams.csv')
m_reg = pd.read_csv(DATA_PATH + 'MRegularSeasonCompactResults.csv')
m_tourney = pd.read_csv(DATA_PATH + 'MNCAATourneyCompactResults.csv')
m_seeds = pd.read_csv(DATA_PATH + 'MNCAATourneySeeds.csv')
m_rankings = pd.read_csv(DATA_PATH + 'MMasseyOrdinals.csv')  

# Submission file stuff
sample_sub_stage1 = pd.read_csv(DATA_PATH + 'SampleSubmissionStage1.csv')
sample_sub_stage2 = pd.read_csv(DATA_PATH + 'SampleSubmissionStage2.csv')

m_reg = m_reg[m_reg['Season'] >= 2003]
m_tourney = m_tourney[m_tourney['Season'] >= 2003]
m_seeds = m_seeds[m_seeds['Season'] >= 2003]

m_reg.tail()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
198572,2026,132,1335,88,1463,84,N,1
198573,2026,132,1345,80,1276,72,N,0
198574,2026,132,1378,70,1455,55,N,0
198575,2026,132,1433,70,1173,62,N,0
198576,2026,132,1465,63,1430,61,A,0


In [98]:

w = m_reg[['Season','WTeamID','WScore','LScore']].rename(
    columns={'WTeamID':'TeamID','WScore':'PointsFor','LScore':'PointsAgainst'})
w['Win'] = 1

l = m_reg[['Season','LTeamID','LScore','WScore']].rename(
    columns={'LTeamID':'TeamID','LScore':'PointsFor','WScore':'PointsAgainst'})
l['Win'] = 0

team_games = pd.concat([w, l], ignore_index=True)


#find average for win and points
team_season = team_games.groupby(['Season','TeamID']).agg(
    WinPct=('Win','mean'),
    AvgPointsFor=('PointsFor','mean'),
    AvgPointsAgainst=('PointsAgainst','mean')
).reset_index()

#adding point diff to later be able to find margin in point diff and win percentage 
team_season['PointDiff'] = team_season['AvgPointsFor'] - team_season['AvgPointsAgainst']



In [99]:
#adding seed stuff
m_seeds['SeedNum'] = m_seeds['Seed'].str[1:3].astype(int)

team_season = team_season.merge(m_seeds[['Season','TeamID','SeedNum']], on=['Season','TeamID'], how='left')

team_season['SeedNum'] = team_season['SeedNum'].fillna(17) #fill for the unqualified teams 



In [100]:
m_conf = pd.read_csv(DATA_PATH + 'MTeamConferences.csv') #adding the conference column if strong

m_conf = m_conf[m_conf['Season'] >= 2003]
power_confs = ['big_ten', 'acc', 'sec', 'big_twelve', 'big_east', 'pac_twelve']
m_conf['PowerConf'] = m_conf['ConfAbbrev'].isin(power_confs).astype(int)

team_season = team_season.merge(m_conf[['Season','TeamID','PowerConf']], on=['Season','TeamID'], how='left')

team_season['PowerConf'] = team_season['PowerConf'].fillna(0)

In [101]:
def get_massey(ordinals_df, season):
    so = ordinals_df[ordinals_df['Season'] == season]
    max_day = so['RankingDayNum'].max()
    late = so[so['RankingDayNum'] >= max_day - 14]
    return late.groupby('TeamID')['OrdinalRank'].agg(MasseyMean='mean', MasseyMedian='median', MasseyMin='min').reset_index()

massey_all = pd.concat([
    get_massey(m_rankings, season).assign(Season=season)
    for season in m_rankings['Season'].unique() if season >= 2003], ignore_index=True)

team_season = team_season.merge(massey_all, on=['Season','TeamID'], how='left')

team_season['MasseyMean'] = team_season['MasseyMean'].fillna(200)
team_season['MasseyMedian'] = team_season['MasseyMedian'].fillna(200)
team_season['MasseyMin'] = team_season['MasseyMin'].fillna(200)

In [102]:
#include tournament data now 


w_tour = m_tourney[['Season','WTeamID','LTeamID']].rename(
    columns={'WTeamID':'TeamID','LTeamID':'OpponentID'})
w_tour['Outcome'] = 1

l_tour = m_tourney[['Season','LTeamID','WTeamID']].rename(
    columns={'LTeamID':'TeamID','WTeamID':'OpponentID'})
l_tour['Outcome'] = 0

matchups = pd.concat([w_tour, l_tour], ignore_index=True)

matchups.head()

,Season,TeamID,OpponentID,Outcome
0,2003,1421,1411,1
1,2003,1112,1436,1
2,2003,1113,1272,1
3,2003,1141,1166,1
4,2003,1143,1301,1


In [103]:
team_stats = team_season[['Season','TeamID','WinPct','PointDiff','SeedNum','PowerConf','MasseyMean','MasseyMedian','MasseyMin']]

#combining regular season and tourn

matchups = matchups[['Season','TeamID','OpponentID','Outcome']]  # strip any old stat columns first
matchups = matchups.merge(team_stats, on=['Season','TeamID'], how='left')
matchups = matchups.merge(team_stats, left_on=['Season','OpponentID'], right_on=['Season','TeamID'],
                           suffixes=('','_opp'), how='left')

matchups.head()



,Season,TeamID,OpponentID,Outcome,WinPct,PointDiff,SeedNum,PowerConf,MasseyMean,MasseyMedian,MasseyMin,TeamID_opp,WinPct_opp,PointDiff_opp,SeedNum_opp,PowerConf_opp,MasseyMean_opp,MasseyMedian_opp,MasseyMin_opp
0,2003,1421,1411,1,0.448276,-7.241379,16.0,0,247.566667,245.5,184.0,1411,0.600000,1.966667,16.0,0,251.522222,252.0,147.0
1,2003,1112,1436,1,0.892857,14.964286,1.0,0,1.802083,1.5,1.0,1436,0.655172,4.655172,16.0,0,165.355556,169.0,101.0
2,2003,1113,1272,1,0.620690,6.793103,10.0,0,34.800000,33.5,22.0,1272,0.793103,8.689655,7.0,0,23.073684,23.0,7.0
3,2003,1141,1166,1,0.793103,6.103448,11.0,0,53.377778,50.5,12.0,1166,0.878788,14.909091,6.0,0,24.156250,21.5,8.0
4,2003,1143,1301,1,0.724138,4.724138,8.0,0,31.638298,30.0,19.0,1301,0.600000,4.400000,9.0,1,53.411111,52.0,23.0


In [104]:
#find margins between team stats for each matchup 

matchups['WinPctMargin'] = matchups['WinPct'] - matchups['WinPct_opp']
matchups['PointDiffMargin'] = matchups['PointDiff'] - matchups['PointDiff_opp']
matchups['SeedMargin'] = matchups['SeedNum'] - matchups['SeedNum_opp']
matchups['PowerConfMargin'] = matchups['PowerConf'] - matchups['PowerConf_opp']
matchups['MasseyMeanMargin'] = matchups['MasseyMean'] - matchups['MasseyMean_opp']
matchups['MasseyMedianMargin'] = matchups['MasseyMedian'] - matchups['MasseyMedian_opp']
matchups['MasseyMinMargin'] = matchups['MasseyMin'] - matchups['MasseyMin_opp']
val_seasons = [2023,2024, 2025]

final = matchups[['Season','TeamID','OpponentID',
                   'WinPctMargin','PointDiffMargin','SeedMargin','PowerConfMargin',
                   'MasseyMeanMargin','MasseyMedianMargin','MasseyMinMargin',
                   'SeedNum','SeedNum_opp',
                   'Outcome']]

final.head()

#making the seed matchup table (improved log reg score by 0.003) not substantial  

train_only = final[~final['Season'].isin(val_seasons)]


seed_pair_stats = train_only.groupby(['SeedNum','SeedNum_opp'])['Outcome'].agg(
    SeedPairWinRate='mean', SeedPairGames='count'
).reset_index()


overall_rate = train_only['Outcome'].mean()
smoothing_strength = 10

seed_pair_stats['SeedPairWinRate_smoothed'] = (
    (seed_pair_stats['SeedPairWinRate'] * seed_pair_stats['SeedPairGames'] + overall_rate * smoothing_strength)
    / (seed_pair_stats['SeedPairGames'] + smoothing_strength)
)


final = final.merge(
    seed_pair_stats[['SeedNum','SeedNum_opp','SeedPairWinRate_smoothed']],
    on=['SeedNum','SeedNum_opp'], how='left'
)

final['SeedPairWinRate_smoothed'] = final['SeedPairWinRate_smoothed'].fillna(overall_rate)

In [105]:

train = final[~final['Season'].isin(val_seasons)]
val = final[final['Season'].isin(val_seasons)]

In [106]:
feature_cols = ['WinPctMargin','PointDiffMargin','SeedMargin','PowerConfMargin',
                 'MasseyMeanMargin','MasseyMedianMargin','MasseyMinMargin','SeedPairWinRate_smoothed']

log_reg = LogisticRegression(max_iter=2000)
log_reg.fit(train[feature_cols], train['Outcome'])

LogisticRegression(max_iter=2000)

In [107]:
gb = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss',
                        max_depth=5, n_estimators=200, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8)
gb.fit(train[feature_cols], train['Outcome'])
probs_gb = gb.predict_proba(val[feature_cols])[:, 1]
print("GB with 7 features:", brier_score_loss(val['Outcome'], probs_gb))

GB with 7 features: 0.19616772607086025


In [108]:
probs = log_reg.predict_proba(val[feature_cols])[:, 1]
score = brier_score_loss(val['Outcome'], probs)
print("Logistic Regression Brier score:", score)

Logistic Regression Brier score: 0.18527743040533942


In [109]:
# ── Load women's files ──
w_teams = pd.read_csv(DATA_PATH + 'WTeams.csv')
w_reg = pd.read_csv(DATA_PATH + 'WRegularSeasonCompactResults.csv')
w_tourney = pd.read_csv(DATA_PATH + 'WNCAATourneyCompactResults.csv')
w_seeds = pd.read_csv(DATA_PATH + 'WNCAATourneySeeds.csv')

# ── Restrict to 2003+ ──
w_reg = w_reg[w_reg['Season'] >= 2003]
w_tourney = w_tourney[w_tourney['Season'] >= 2003]
w_seeds = w_seeds[w_seeds['Season'] >= 2003]

# ── Reshape into one row per team per game ──
w_w = w_reg[['Season','WTeamID','WScore','LScore']].rename(
    columns={'WTeamID':'TeamID','WScore':'PointsFor','LScore':'PointsAgainst'})
w_w['Win'] = 1

w_l = w_reg[['Season','LTeamID','LScore','WScore']].rename(
    columns={'LTeamID':'TeamID','LScore':'PointsFor','WScore':'PointsAgainst'})
w_l['Win'] = 0

w_team_games = pd.concat([w_w, w_l], ignore_index=True)

# ── Average win % and point differential per team per season ──
w_team_season = w_team_games.groupby(['Season','TeamID']).agg(
    WinPct=('Win','mean'),
    AvgPointsFor=('PointsFor','mean'),
    AvgPointsAgainst=('PointsAgainst','mean')
).reset_index()

w_team_season['PointDiff'] = w_team_season['AvgPointsFor'] - w_team_season['AvgPointsAgainst']

# ── Add seed ──
w_seeds['SeedNum'] = w_seeds['Seed'].str[1:3].astype(int)
w_team_season = w_team_season.merge(w_seeds[['Season','TeamID','SeedNum']], on=['Season','TeamID'], how='left')
w_team_season['SeedNum'] = w_team_season['SeedNum'].fillna(17)

# ── Build matchup table from w_tourney ──
w_w_tour = w_tourney[['Season','WTeamID','LTeamID']].rename(
    columns={'WTeamID':'TeamID','LTeamID':'OpponentID'})
w_w_tour['Outcome'] = 1

w_l_tour = w_tourney[['Season','LTeamID','WTeamID']].rename(
    columns={'LTeamID':'TeamID','WTeamID':'OpponentID'})
w_l_tour['Outcome'] = 0

w_matchups = pd.concat([w_w_tour, w_l_tour], ignore_index=True)

# ── Merge in team and opponent stats ──
w_team_stats = w_team_season[['Season','TeamID','WinPct','PointDiff','SeedNum']]

w_matchups = w_matchups.merge(w_team_stats, on=['Season','TeamID'], how='left')
w_matchups = w_matchups.merge(w_team_stats, left_on=['Season','OpponentID'], right_on=['Season','TeamID'],
                               suffixes=('','_opp'), how='left')

# ── Compute margins ──
w_matchups['WinPctMargin'] = w_matchups['WinPct'] - w_matchups['WinPct_opp']
w_matchups['PointDiffMargin'] = w_matchups['PointDiff'] - w_matchups['PointDiff_opp']
w_matchups['SeedMargin'] = w_matchups['SeedNum'] - w_matchups['SeedNum_opp']

w_final = w_matchups[['Season','TeamID','OpponentID','WinPctMargin','PointDiffMargin','SeedMargin','Outcome']]

# ── Sanity check ──
print(w_final.shape)
print(w_final.isnull().sum())

# ── Split by season (same 3-season window as men's) ──
val_seasons = [2023, 2024, 2025]
w_train = w_final[~w_final['Season'].isin(val_seasons)]
w_val = w_final[w_final['Season'].isin(val_seasons)]

feature_cols = ['WinPctMargin', 'PointDiffMargin', 'SeedMargin']

# ── Train logistic regression ──
log_reg_w = LogisticRegression(max_iter=2000)
log_reg_w.fit(w_train[feature_cols], w_train['Outcome'])

probs_w = log_reg_w.predict_proba(w_val[feature_cols])[:, 1]
score_w = brier_score_loss(w_val['Outcome'], probs_w)
print("Women's Logistic Regression Brier score:", score_w)

(2804, 7)
Season             0
TeamID             0
OpponentID         0
WinPctMargin       0
PointDiffMargin    0
SeedMargin         0
Outcome            0
dtype: int64
Women's Logistic Regression Brier score: 0.13164832273894417
